#### 1. Voxelization

##### (1) Voxelization with labels

###### Voxelize point clouds with leaf/wood labels for training

In [ ]:
import os
import pandas as pd
import numpy as np

def voxel_split_8_with_label(df: pd.DataFrame):
    """
    Input : Dataframe with x, y, z, label column 
    Output : Voxelized dataframe list [(x, y, z, label)]
    """
    xyz = df.iloc[:, :3].values
    labels = df.iloc[:, 7].values.reshape(-1, 1)  # label

    # 전체 bbox 기준 mid 계산
    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        label_sub = labels[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame()) 
        else:
            combined = np.hstack([xyz_sub, label_sub])
            voxel_dfs.append(pd.DataFrame(combined))

    return voxel_dfs

def voxel_add_split_8_with_label(df: pd.DataFrame):
    """
    Input : Dataframe with x, y, z, label column 
    Output : Voxelized dataframe list [(x, y, z, label)]
    """
    xyz = df.iloc[:, :3].values
    labels = df.iloc[:, 7].values.reshape(-1, 1)  # iloc should be label column

    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        label_sub = labels[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame())
        else:
            combined = np.hstack([xyz_sub, label_sub])
            voxel_dfs.append(pd.DataFrame(combined))

    return voxel_dfs


def split_large_files_in_dir(target_dir, threshold=100000):
    files = sorted(f for f in os.listdir(target_dir) if f.endswith(".csv"))
    print(f"[INFO] Check {len(files)} files from {target_dir}")

    for f in files:
        path = os.path.join(target_dir, f)
        df = pd.read_csv(path, header=None)
        num_points = len(df)

        if num_points >= threshold:
            base_num = int(os.path.splitext(f)[0])
            print(f"[SPLIT] {f} ({num_points} points) → Voxelization")

            voxel_dfs = voxel_add_split_8_with_label(df) 

            for i, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num + (i+1)*10:08d}.csv"
                out_path = os.path.join(target_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

            os.remove(path)
            print(f"    └ Deleted original file: {f}")


def process_dir(in_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    files = sorted(f for f in os.listdir(in_dir) if f.endswith(".csv"))

    print(f"[INFO] {in_dir} → {out_dir} processing start (total file num : {len(files)})")

    for f in files:
        file_id = os.path.splitext(f)[0]
        base_num = int(file_id)
        in_path = os.path.join(in_dir, f)

        try:
            df = pd.read_csv(in_path)
            voxel_dfs = voxel_split_8_with_label(df)

            counts = [len(voxel) for voxel in voxel_dfs]
            max_pts = max(counts)
            min_pts = min(counts)

            print(f"[INFO] {f}: voxel point counts = {counts} → max = {max_pts}, min = {min_pts}")

            for j, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue

                out_name = f"{base_num + (j+1):08d}.csv"
                out_path = os.path.join(out_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)

        except Exception as e:
            print(f"[ERROR] {f}: {e}")

# === 실행 파트 ===
BASE_IN = "/your/original/data/path"
BASE_OUT = "/your/output/path"

for subdir in ["yoursubdir1", "yoursubdir2"]: # put your category folder according to synsetoffset2category.txt 
    in_path = os.path.join(BASE_IN, subdir)
    out_path = os.path.join(BASE_OUT, subdir)
    process_dir(in_path, out_path)
    split_large_files_in_dir(out_path, threshold=100000)

print("\n Preprocessing completed")


##### (2) Voxelization without labels

###### Voxelize point clouds without leaf/wood labels for inference

In [ ]:
import os
import pandas as pd
import numpy as np

def voxel_split_8(df: pd.DataFrame):
    xyz = df.iloc[:, :3].values

    return _voxel_split_core(xyz)

def voxel_split_8_noshift(df: pd.DataFrame):
    xyz = df.iloc[:, :3].values
    return _voxel_split_core(xyz)

def _voxel_split_core(xyz: np.ndarray):
    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame())
        else:
            voxel_dfs.append(pd.DataFrame(xyz_sub))
    return voxel_dfs

def process_dir(in_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    files = sorted(f for f in os.listdir(in_dir) if f.endswith(".txt"))

    print(f"[INFO] Start Processing {in_dir} → {out_dir} (Total file num : {len(files)})")
    for f in files:
        file_id = os.path.splitext(f)[0]
        try:
            base_num = int(file_id)
        except:
            print(f"[SKIP] File name {f} → Can not convert to number")
            continue

        in_path = os.path.join(in_dir, f)
        try:
            df = pd.read_csv(in_path, sep=None, engine="python", header=None)
            
            if df.shape[0] == 0:
                print(f"[SKIP] {f}: no valid numeric rows")
                continue

            voxel_dfs = voxel_split_8(df) 

            counts = [len(voxel) for voxel in voxel_dfs]
            print(f"[INFO] {f}: voxel point counts = {counts}")

            for j, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num*10000 + (j+1):08d}.csv"
                out_path = os.path.join(out_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

        except Exception as e:
            print(f"[ERROR] {f}: {e}")

def split_large_files_in_dir(target_dir, threshold=600000):
    files = sorted(f for f in os.listdir(target_dir) if f.endswith(".csv"))
    print(f"[INFO] Check {len(files)} from {target_dir}")

    for f in files:
        path = os.path.join(target_dir, f)
        df = pd.read_csv(path, header=None)
        num_points = len(df)

        if num_points >= threshold:
            base_num = int(os.path.splitext(f)[0])
            print(f"[SPLIT] {f} ({num_points} points) → Voxelization")

            voxel_dfs = voxel_split_8_noshift(df) 

            for i, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num + (i+1)*10:08d}.csv"
                out_path = os.path.join(target_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

            os.remove(path)
            print(f"    └ Deleted original file: {f}")


BASE_IN = "/Input/File/Direction"
BASE_OUT = "/Output/File/Direction/00000000"

process_dir(BASE_IN, BASE_OUT)
split_large_files_in_dir(BASE_OUT, threshold=600000)

print("\n Processing completed.")

##### Large file split (additional)

###### Split large file for GPU memory limitation

In [ ]:
import os
import pandas as pd
import numpy as np

def voxel_split_8_with_label(df: pd.DataFrame):
    """
    Input : Dataframe with x, y, z, label column 
    Output : Voxelized dataframe list [(x, y, z, label)]
    """
    xyz = df.iloc[:, :3].values
    labels = df.iloc[:, 3].values.reshape(-1, 1)  # label
    # 전체 bbox 기준 mid 계산
    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        label_sub = labels[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame())  # 빈 DataFrame
        else:
            combined = np.hstack([xyz_sub, label_sub])
            voxel_dfs.append(pd.DataFrame(combined))

    return voxel_dfs


def split_large_files_in_dir(target_dir, threshold=100000):
    files = sorted(f for f in os.listdir(target_dir) if f.endswith(".csv"))
    print(f"[INFO] Check {len(files)} from {target_dir}")

    for f in files:
        path = os.path.join(target_dir, f)
        df = pd.read_csv(path, header=None)
        num_points = len(df)

        if num_points >= threshold:
            base_num = int(os.path.splitext(f)[0])
            print(f"[SPLIT] {f} ({num_points} points) → Voxelization")

            voxel_dfs = voxel_split_8_with_label(df)

            for i, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num + (i+1)*1000:08d}.csv"
                out_path = os.path.join(target_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

            # 원본 파일 삭제
            os.remove(path)
            print(f"    └ Deleted original file: {f}")


# 실행
target_dir = "/target/dir"
split_large_files_in_dir(target_dir, threshold=100000)

print("\n✅ Processing Completed")

##### (4) Small file delete

###### Delete small file for preventing grouping error

In [ ]:
import os
from pathlib import Path

# 대상 디렉토리
target_dir = Path("/target/dir")

# 기준 크기 (1KB = 1024 바이트)
threshold = 1024  

# 디렉토리 내 모든 csv 파일 검사
for file in target_dir.glob("*.csv"):
    size = os.path.getsize(file)
    if size < threshold:
        print(f"Deleting {file} (size={size} bytes)")
        os.remove(file)

print("✅ Deletion completed")


#### Dataset Direction

###### create .json for loading data

##### Create Training dataset

In [ ]:
import os
import json
import random
from collections import defaultdict

# === Directory settings (replace with your actual dataset paths) ===
bl_dir = "dataset/LWSEG_Voxel2/BL"   # Class BL directory
nl_dir = "dataset/LWSEG_Voxel2/NL"   # Class NL directory

# Output directory for JSON splits
output_dir = "dataset/LWSEG_Voxel2/json_splits"
os.makedirs(output_dir, exist_ok=True)

# Collect all files and group by the first 4 characters of the filename
file_groups = defaultdict(list)

def collect_files(dir_path, prefix):
    for fname in os.listdir(dir_path):
        if fname.endswith(".csv"):
            group_key = fname[:4]  # group by first 4 characters
            path = f"LWSEG_Voxel2/{prefix}/{os.path.splitext(fname)[0]}"
            file_groups[group_key].append(path)

# Collect files from both classes
collect_files(bl_dir, "BL")
collect_files(nl_dir, "NL")

# Shuffle group keys
group_keys = list(file_groups.keys())
random.shuffle(group_keys)

# Split into train:test:val:inference = 7:1:1:1 ratio
n = len(group_keys)
train_keys = group_keys[: int(n * 0.7)]
test_keys  = group_keys[int(n * 0.7) : int(n * 0.8)]
val_keys   = group_keys[int(n * 0.8) : int(n * 0.9)]
inf_keys   = group_keys[int(n * 0.9) :]

# Build file path lists based on group keys
train_paths = [path for key in train_keys for path in file_groups[key]]
test_paths  = [path for key in test_keys  for path in file_groups[key]]
val_paths   = [path for key in val_keys   for path in file_groups[key]]
inf_paths   = [path for key in inf_keys   for path in file_groups[key]]

# Save JSON utility
def save_json(data, filename):
    with open(os.path.join(output_dir, filename), "w") as f:
        json.dump(data, f, indent=4)

# Save JSON splits
save_json(train_paths, "leafwood_data_train.json")
save_json(test_paths,  "leafwood_data_test.json")
save_json(val_paths,   "leafwood_data_val.json")
save_json(inf_paths,   "leafwood_data_inference.json")

# Summary
print(f"Train groups: {len(train_keys)}  files: {len(train_paths)}")
print(f"Test  groups: {len(test_keys)}  files: {len(test_paths)}")
print(f"Val   groups: {len(val_keys)}  files: {len(val_paths)}")
print(f"Infer groups: {len(inf_keys)}  files: {len(inf_paths)}")
print(f"JSON files saved to → {output_dir}")


##### Create Inference dataset 

In [ ]:
import os
import json
import random
from collections import defaultdict

# === Input directory (replace with your dataset path) ===
data_dir = "dataset/LWSEG_Voxel2"

# === Output directory for JSON splits ===
output_dir = "dataset/LWSEG_Voxel2/json_grouped"
os.makedirs(output_dir, exist_ok=True)

# Group files by the first 4 characters of the filename
file_groups = defaultdict(list)

def collect_files(dir_path, prefix):
    for fname in os.listdir(dir_path):
        if fname.endswith(".csv"):
            group_key = fname[:4]  # group by first 4 characters
            path = f"EVAL_data_Voxel/{prefix}/{os.path.splitext(fname)[0]}"
            file_groups[group_key].append(path)

# Collect files (example: prefix = class ID or folder name)
collect_files(data_dir, "CLASS_ID")

# Shuffle group keys
group_keys = list(file_groups.keys())
random.shuffle(group_keys)

# Split into train:test:val:inference = 7:1:1:1 ratio
n = len(group_keys)
train_keys = group_keys[: int(n * 0.7)]
test_keys  = group_keys[int(n * 0.7) : int(n * 0.8)]
val_keys   = group_keys[int(n * 0.8) : int(n * 0.9)]
inf_keys   = group_keys[:]   # use all keys for inference

# Build file path lists
train_paths = [path for key in train_keys for path in file_groups[key]]
test_paths  = [path for key in test_keys  for path in file_groups[key]]
val_paths   = [path for key in val_keys   for path in file_groups[key]]
inf_paths   = [path for key in inf_keys   for path in file_groups[key]]

# Save JSON helper
def save_json(data, filename):
    with open(os.path.join(output_dir, filename), "w") as f:
        json.dump(data, f, indent=4)

# Save JSON splits
save_json(train_paths, "leafwood_data_train.json")
save_json(test_paths,  "leafwood_data_test.json")
save_json(val_paths,   "leafwood_data_val.json")
save_json(inf_paths,   "leafwood_data_inference.json")

# Summary
print(f"Train groups: {len(train_keys)}  files: {len(train_paths)}")
print(f"Test  groups: {len(test_keys)}  files: {len(test_paths)}")
print(f"Val   groups: {len(val_keys)}  files: {len(val_paths)}")
print(f"Infer groups: {len(inf_keys)}  files: {len(inf_paths)}")
print(f"JSON files saved to → {output_dir}")


#### Synsetoffset2category.txt generation

###### Generate Category txt file
###### It has to locate same direction with classified_json_grouped and each data folder

In [ ]:
from pathlib import Path

# Save directory and file name (update the path as needed)
save_dir = Path("dataset/LWSEG_Voxel2")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "synsetoffset2category.txt"

# Category to folder mapping (example)
content = """NL 22222222
BL 33333333
"""

# Write content to file
with open(save_path, "w", encoding="utf-8") as f:
    f.write(content)

print(f"✅ File saved at: {save_path}")


### Merging Voxels

###### Merge Voxels through group ids.
###### Merged .csv file will be created in src_dir/merged/ folder

In [ ]:
import os
import pandas as pd
from collections import defaultdict

# Input and output paths (update as needed)
src_dir = 'dataset/Model_B_inf/00000000'
output_dir = os.path.join(src_dir, 'merged')
os.makedirs(output_dir, exist_ok=True)

# Group files by the first 4 characters of filename
grouped_files = defaultdict(list)

for filename in os.listdir(src_dir):
    if filename.endswith('.csv'):
        key = filename[:4]  # group by first 4 digits
        grouped_files[key].append(filename)

# Merge files vertically for each group
for key, files in grouped_files.items():
    dfs = []
    for file in sorted(files):
        path = os.path.join(src_dir, file)
        df = pd.read_csv(path, header=None)  # no header in input
        dfs.append(df)

    merged_df = pd.concat(dfs, axis=0, ignore_index=True)  # vertical merge
    output_path = os.path.join(output_dir, f"{key}.csv")
    merged_df.to_csv(output_path, index=False, header=False)  # no header in output

    print(f"Saved: {output_path} with {len(merged_df)} rows")
